In [1]:
import os
import sys

if os.path.join(os.path.realpath("."), "submodules") not in sys.path:
    sys.path.append(os.path.join(os.path.realpath("."), "submodules"))
print(sys.path)

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/lib/python3.12/site-packages', '/mnt/internal/LinuxData/vc/crypto/isd-leda/isdleda/examples/submodules']


In [5]:
import csv
proper_primes = None
#with open('proper_primes.csv', 'r', newline='') as csvfile:
with open('../assets/proper_primes.csv', 'r', newline='') as csvfile:
    reader = csv.reader(csvfile)
    # actually there's just one row
    for row in reader:
        proper_primes = list(map(int, row))

In [8]:
import itertools
import numpy as np
from dataclasses import dataclass, field, asdict
from enum import StrEnum, auto
from typing import Optional, Set, Any, Dict

In [9]:
class Attacks(StrEnum):
    KeyR1 = auto()
    KeyR2 = auto()
    KeyR3 = auto()
    MsgR = auto()

@dataclass(eq=True, frozen=True, order=True)
class Value:
    n: int
    r: int
    t: int
    # k: int = field(init=False, compare=False)
    prime: Optional[int] = field(default=None, compare=False)
    n0: Optional[int] = field(default=None, compare=False)
    v: Optional[int] = field(default=None, compare=False)
    lambd: Optional[int] = field(default=None, compare=False)


@dataclass(order=True)
class ISDVariant:    
    attack: Attacks = field(compare=False)
    isd_variant: str
    isd_variant_options: Dict[str, Any]
    value: Value


In [64]:
n0_values = range(2, 6)
lambda_values = (128, 192, 256)
values: Set[Value] = set()
    
for n0, lam in itertools.product(n0_values, lambda_values):
    # print(n0, lam)
    # ISD(n_0*p,p,t) / sqrt(p) attacked code rate (n_0-1)/n_0 
    t1 = np.ceil(-lam / np.log2(1-(n0-1)/n0) )
    # ISD(n_0*p,p,2*v) / (p* binom{n_0}{2}), attacked code rate (n_0-1)/n_0
    v1 = np.ceil(-lam / np.log2(1-(n0-1)/n0) ) // 2
    # ISD(2*p,p,2*v) / (n_0*p), attacked code rate (1/2)
    v2 = np.ceil(-lam / np.log2(1-1/2) ) // 2
    # ISD(n_0*p,(n_0-1)*p,n_0*v) / p    attacked code rate 1/n_0
    v3 = np.ceil(-lam/ np.log2(1-1/n0)) // n0
    
    # KEY recovery
    v_min = min(v1, v2, v3)
    v_max = max(v1, v2, v3)

    # v should be odd
    v_range_low = int(.8 * v_min)
    v_range_high = int(1.3 * v_min)
    if not v_range_low % 2:
        v_range_low -= 1
    if not v_range_high % 2:
        v_range_high += 1

    for v in range(v_range_low, v_range_high, 2):
        # p*n0 = (v*n0)^2 -> p = v^2*n0
        prime_guess = v**2 * n0
        prime_range = filter(lambda p: p>= int(.8 * prime_guess) and p <= int(
            1.2 * prime_guess), proper_primes)
        for prime in prime_range:
            # key recovery 1 (n0*p, p, 2*v)
            value = Value(n = prime * n0, r=prime, t=2*v, prime=prime, n0=n0, v=v, lambd=lam)
            values.add(value)

            if n0 == 2:
                # key recovery 2 (2p, p, 2v)
                value = Value(n = prime * 2, r=prime, t=2*v, prime=prime, n0=n0, v=v, lambd=lam)
                values.add(value)

            # key recovery 3 (n0*p, (n0-1)*p, n0*v
            value = Value(n = prime * n0, r=prime, t=2*v, prime=prime, n0=n0, v=v, lambd=lam)
            values.add(value)
            # classical = min(kr1, kr2, kr3)

    # MSG recovery
    for t in range(int(t1 * .8), int(t1 * 1.2)):
        # t = 2v 
        prime_guess = (t/2)**2 * n0
        prime_range = filter(lambda p: p>= int(.8 * prime_guess) and p <= int(
            1.2 * prime_guess), proper_primes)
        for prime in prime_range:
            # msg recovery p*n0, p, t
            value = Value(n = prime * n0, r=prime, t=2*v, prime=prime, n0=n0, lambd=lam)
            values.add(value)


{Value(n=25287, r=8429, t=98, prime=8429, n0=3), Value(n=51206, r=25603, t=222, prime=25603, n0=2), Value(n=84236, r=21059, t=134, prime=21059, n0=4), Value(n=100652, r=25163, t=158, prime=25163, n0=4), Value(n=45062, r=22531, t=154, prime=22531, n0=4), Value(n=113062, r=56531, t=310, prime=56531, n0=2), Value(n=40538, r=20269, t=190, prime=20269, n0=2), Value(n=17338, r=8669, t=78, prime=8669, n0=5), Value(n=88106, r=44053, t=278, prime=44053, n0=2), Value(n=63062, r=31531, t=246, prime=31531, n0=2), Value(n=68396, r=17099, t=146, prime=17099, n0=4), Value(n=42538, r=21269, t=138, prime=21269, n0=4), Value(n=49094, r=24547, t=218, prime=24547, n0=2), Value(n=50815, r=10163, t=98, prime=10163, n0=5), Value(n=15692, r=3923, t=58, prime=3923, n0=4), Value(n=105513, r=35171, t=202, prime=35171, n0=3), Value(n=54214, r=27107, t=198, prime=27107, n0=3), Value(n=79671, r=26557, t=202, prime=26557, n0=3), Value(n=87303, r=29101, t=194, prime=29101, n0=3), Value(n=79978, r=39989, t=274, prime=